# Exercise Solution — Transaction Monitor: Consumer

Consumes transactions from the `transactions` topic and:
- **Part A** — Classifies each message as OK or ALERT and writes alerts to a log file
- **Part B** — Tracks per-account stats and prints a running summary every 10 messages

In [ ]:
from kafka import KafkaConsumer
import json

# ----- Configuration -----
TOPIC            = 'transactions'
BROKERS          = ['course-kafka:9092']
ALERTS_FILE      = '/home/developer/kafka/sinkFiles/alerts.log'
ALERT_THRESHOLD  = 1000    # ILS — any transaction above this is flagged
SUMMARY_EVERY    = 10      # print a stats summary every N messages

In [ ]:
# group_id='monitors' — all consumers sharing this group id split the partitions
# between them. Since account_id is the message key, all transactions from the
# same account always go to the same partition → same consumer instance.
#
# auto_commit_interval_ms=1000 — the consumer automatically saves (commits)
# its current offset to Kafka every second, so on restart it picks up
# exactly where it left off instead of re-reading from the beginning.
consumer = KafkaConsumer(
    TOPIC,
    group_id                = 'monitors',
    bootstrap_servers       = BROKERS,
    auto_commit_interval_ms = 1000
)

In [ ]:
# ----- Part B: in-memory stats (initialised before the loop) -----
# Using plain dicts is enough here — no external storage needed.
account_counts  = {}   # { 'ACC_1': 5, ... }
account_totals  = {}   # { 'ACC_1': 4821.50, ... }
alert_count     = 0
total_messages  = 0


def print_summary():
    """Print a formatted stats table for the current session."""
    print(f"\n{'='*50}")
    print(f" Summary (after {total_messages} messages)")
    print(f"{'='*50}")
    for acc in sorted(account_counts):
        print(f"  {acc} | {account_counts[acc]:>2} transactions | "
              f"total: {account_totals[acc]:>10,.2f} ILS")
    rate = (alert_count / total_messages) * 100
    print(f"  Alert rate: {rate:.1f}%  ({alert_count} out of {total_messages})")
    print(f"{'='*50}\n")

In [ ]:
# Open the alerts file once for the lifetime of the consumer.
# More efficient than opening/closing on every message.
with open(ALERTS_FILE, 'w') as f:

    for message in consumer:

        # ----- Deserialize -----
        # The producer sent json.dumps(...).encode('utf-8'), so we reverse that:
        # first decode bytes → str, then parse str → dict.
        data   = json.loads(message.value.decode('utf-8'))
        acc    = data['account_id']
        amount = data['amount']
        city   = data['city']

        total_messages += 1

        # ----- Part B: update stats -----
        account_counts[acc] = account_counts.get(acc, 0) + 1
        account_totals[acc] = account_totals.get(acc, 0.0) + amount

        # ----- Part A: classify -----
        if amount > ALERT_THRESHOLD:
            alert_count += 1
            alert_line = f"ALERT | {acc} | {amount:.2f} ILS | {city}"
            print(f"🚨 {alert_line}")

            # Write only alerts to the log file
            f.write(alert_line + '\n')

            # flush() forces the OS write buffer to disk immediately.
            # Without this, alerts may be lost if the consumer crashes
            # before the buffer fills up naturally.
            f.flush()

        else:
            print(f"✅ OK    | {acc} | {amount:.2f} ILS")

        # ----- Part B: print summary every N messages -----
        if total_messages % SUMMARY_EVERY == 0:
            print_summary()